# Zero-Shot Prompting — Getting Great Results With No Examples

In [ ]:
# If you are running this on Google Colab, uncomment and run the line below first.
# !pip install -q boto3 openai tiktoken anthropic matplotlib

## How LLM calls work in this notebook

Every live API call goes through `LLMRouter` from `garage_helper`. Here is what happens under the hood:

1. **Model resolution** — `LLMRouter` looks up your model in its registry and maps it to the right provider (`anthropic`, `openai`, `azure_openai`, `bedrock_claude`, `gemini`, etc.). `setup_llm()` sets this up once for the session.
2. **Provider instantiation** — the matching provider class is created and cached. The underlying API client is reused across all cells — no reconnection overhead.
3. **Request dispatch** — `router.generate(prompt, system=..., **kwargs)` forwards the prompt to the provider API and returns the reply as a plain string.
4. **Logging** — flip `verbose=True` on the router to see every request logged with token counts directly in the cell output.

These notebooks have been tested with **Claude** (via Anthropic direct API and AWS Bedrock) and **GPT** models (via Azure OpenAI and direct OpenAI). Swap `MODEL` in the setup cell to use a different provider without changing anything else.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath("../.."))

from garage_helper import setup_llm, LLMRouter

# Contributors: add DEFAULT_LLM_MODEL (and any provider credentials) to a .env at the repo root.
# Learners: setup_llm() will run an interactive wizard to pick a provider and enter credentials.
MODEL  = setup_llm()
router = LLMRouter(default_model=MODEL, verbose=False)

## You already have everything the model needs — you just have to say it clearly

In the previous notebook — Anatomy of a Prompt — we saw that a prompt is a structured list of messages and that small wording changes flip the output. Zero-shot prompting is where that insight pays off.

"Zero-shot" means you give the model a task with zero examples of what a good answer looks like. No demonstrations, no sample inputs and outputs — just your words and the model's training.

Here is the thing most people get wrong: they assume zero-shot means "just ask the question." It does not. Zero-shot means no examples. It says nothing about whether your prompt is vague or precise, short or structured, constrained or open-ended.

The gap between a bad zero-shot prompt and a great one is rarely about adding examples. It is about five things: **specificity**, **task framing**, **role assignment**, **constraints**, and **what to avoid**. Master those and you will get production-quality results without a single example in your prompt. This is the bread-and-butter skill of working with LLMs.

## Concept 1 — Specificity: the model defaults to the median

When you ask a vague question, the model gives you the most statistically average answer for that question — the response that fits the largest number of possible interpretations. That is rarely what you wanted.

**The analogy:** ordering at a restaurant by saying "bring me something good." The waiter will bring you their most popular dish — safe, inoffensive, chosen for the crowd. But if you say "something spicy, no seafood, under 800 calories" you get something that actually fits you. The kitchen did not change. Your specification did.

Specificity in prompting means giving the model exactly enough information to rule out every answer you do not want. You do not need to be exhaustive — you need to be precise about what distinguishes the answer you want from all the others.

In [ ]:
def ask(prompt: str, max_tokens: int = 250) -> str:
    return router.generate(prompt, model=MODEL, max_tokens=max_tokens)

vague  = "Write a summary of machine learning."

precise = (
    "Write a three-sentence summary of machine learning for a product manager "
    "who has no technical background. Focus on what it enables, not how it works. "
    "Avoid all jargon — if you must use a technical term, define it in plain English immediately."
)

print("=" * 55)
print("VAGUE PROMPT")
print(f"Prompt: '{vague}'")
print("-" * 55)
print(ask(vague))

print()
print("=" * 55)
print("PRECISE PROMPT")
print(f"Prompt: '{precise[:70]}...'")
print("-" * 55)
print(ask(precise))

print()
print("Same model, same knowledge. The specification is doing all the work.")

## Concept 2 — Task framing: lead with the verb

The first word after your opening context matters more than most people realise. The model reads your prompt left-to-right and starts narrowing its output distribution immediately. Verbs do the most work: "explain", "list", "rewrite", "compare", "critique", "classify", "translate" each pull the model in a very different direction before it has read the rest of your sentence.

**The analogy:** imagine giving directions to a driver. "Take the motorway" sent as the first instruction frames everything that follows — speed, exits, lane discipline. If you say "take the backroads" first, the same destination requires completely different decisions. Your task verb is that first instruction. It sets the cognitive mode the model operates in for the rest of the response.

The practical rule: decide what *kind* of output you want before you decide what *content* you want. Then put the task verb first.

In [ ]:
def ask(prompt: str, max_tokens: int = 200) -> str:
    return router.generate(prompt, model=MODEL, max_tokens=max_tokens)

topic = "Python's Global Interpreter Lock (GIL)"

framings = [
    ("Explain",  f"Explain {topic} to a developer who knows Python but has never written concurrent code."),
    ("List",     f"List three practical consequences of {topic} when writing multi-threaded Python."),
    ("Compare",  f"Compare {topic}'s behaviour for CPU-bound vs I/O-bound workloads in two sentences each."),
    ("Critique", f"Critique the design decision behind {topic}. What problem was it solving and what does it cost?"),
]

for verb, prompt in framings:
    response = ask(prompt)
    print(f"[Task verb: {verb}]")
    print(response[:300] + ("..." if len(response) > 300 else ""))
    print("-" * 55)
    print()

print("The topic is identical. The task verb determines the shape of every response.")

## Concept 3 — Role assignment: persona as a configuration dial

Assigning a role to the model is not a gimmick — it is a compact way to specify depth, vocabulary, tone, and assumed background knowledge all at once. Saying "you are a senior security engineer" loads a whole cluster of behaviours: opinionated, specific, focused on practical risk, comfortable with jargon, likely to warn about edge cases.

**The analogy:** think about how differently a GP, a specialist surgeon, and a medical student explain the same diagnosis. Same facts, wildly different vocabulary, depth, and assumed knowledge in the listener. Assigning a role to the model is choosing which of those three sits across the table from you. You do not have to renegotiate the expertise level with every question — the role carries it forward.

A few things make a role assignment stick better: give it a domain ("senior backend engineer"), optionally a context ("at a company with 10M users"), and a stance ("direct and opinionated"). The more specific the role, the tighter the output.

In [ ]:
def ask_with_role(role: str, question: str, max_tokens: int = 200) -> str:
    system = f"You are {role}. Be direct. No preamble."
    return router.generate(question, model=MODEL, system=system, max_tokens=max_tokens)

question = "Should we use a relational database or a document store for our new user profile service?"

roles = [
    "a junior developer one year into your first job",
    "a senior backend engineer at a 5M-user SaaS company with strong opinions on data modelling",
    "a startup CTO who prioritises speed-to-market over technical perfection",
    "a database consultant who charges by the hour and gets called in to fix migrations gone wrong",
]

print(f"Question: '{question}'")
print("=" * 60)

for role in roles:
    response = ask_with_role(role, question)
    short_role = role[:55] + "..." if len(role) > 55 else role
    print(f"\nRole: {short_role}")
    print(response[:280] + ("..." if len(response) > 280 else ""))
    print("-" * 60)

print()
print("The model's knowledge base is the same. The role changes what it surfaces and how.")

## Concept 4 — Stating constraints: give the model guardrails

Constraints are not restrictions — they are information. When you tell the model "respond in exactly three bullet points" or "keep it under 100 words", you are not fighting it. You are giving it information about what success looks like, which makes it easier for the model to produce exactly that.

**The analogy:** a contractor asked to "build something nice" in your backyard will ask you ten clarifying questions. A contractor given a budget, a footprint, and a deadline will start drawing plans. Constraints replace ambiguity with specification. They do not limit quality — they focus it.

The most useful constraint categories are: **length** (word count, sentence count, bullet count), **format** (JSON, markdown, table, code block), **audience** (technical level, role, familiarity), **scope** (what to include, what time period, which aspect), and **tone** (formal, conversational, blunt). You rarely need all five — even one or two removes most of the ambiguity.

In [ ]:
def ask(system: str, user: str, max_tokens: int = 300) -> str:
    return router.generate(user, model=MODEL, system=system, max_tokens=max_tokens)

base_task = "Explain the tradeoffs between REST and GraphQL APIs."
system_base = "You are a helpful assistant."

constraint_experiments = [
    {
        "label":  "No constraints",
        "system": system_base,
        "user":   base_task,
    },
    {
        "label":  "Length + format",
        "system": system_base,
        "user":   base_task + " Respond as a markdown table with columns: Dimension | REST | GraphQL. Maximum 6 rows.",
    },
    {
        "label":  "Audience + scope + tone",
        "system": "You are a direct, opinionated senior engineer. No hedging, no 'it depends' without a concrete tiebreaker.",
        "user":   base_task + " Audience: a backend engineer choosing between the two for a new public API. Give a concrete recommendation at the end.",
    },
]

for exp in constraint_experiments:
    response = ask(exp["system"], exp["user"])
    print(f"[{exp['label']}]")
    print(response[:400] + ("..." if len(response) > 400 else ""))
    print("-" * 60)
    print()

print("Same underlying question. Constraints shaped the length, structure, and usefulness.")

## Concept 5 — Telling the model what NOT to do

This is the most underused technique in zero-shot prompting. Positive instructions tell the model what to aim for. Negative instructions rule out the specific failure modes you have already seen or know to avoid.

**The analogy:** if you are giving someone directions to your house and you know there is a confusing fork where everyone turns left by mistake, you say "stay right at the fork by the petrol station." You do not just say "arrive at my house." Negative instructions are guardrails for the specific wrong turns the model will otherwise take.

Where positive instructions set direction, negative instructions tighten accuracy. Common failure modes worth calling out explicitly: unnecessary preamble ("Great question!", "Certainly!"), restating the question before answering, hedging every statement with "it depends", adding a summary after every response, padding with filler phrases like "in conclusion", or producing more content than requested.

In [ ]:
def ask(system: str, user: str, max_tokens: int = 250) -> str:
    return router.generate(user, model=MODEL, system=system, max_tokens=max_tokens)

user_question = "What is the difference between authentication and authorisation?"

without_negatives = "You are a helpful technical assistant. Be clear and concise."

with_negatives = """You are a helpful technical assistant.
Do not open with affirmations like 'Great question' or 'Certainly'.
Do not restate the question before answering.
Do not add a summary paragraph at the end.
Do not hedge — give direct definitions."""

print("WITHOUT negative instructions:")
print("-" * 55)
r1 = ask(without_negatives, user_question)
print(r1)

print()
print("WITH negative instructions:")
print("-" * 55)
r2 = ask(with_negatives, user_question)
print(r2)

print()
print(f"Without negatives: {len(r1.split())} words")
print(f"With negatives:    {len(r2.split())} words")
print("Negative instructions cut the filler and leave only the signal.")

## Putting it together — a zero-shot prompt template that works

All five concepts wired into a single reusable pattern. The goal is not to memorise a formula but to build a habit: before you hit send, ask yourself whether each dimension is filled in or intentionally left open.

In [ ]:
role = (
    "You are a senior data scientist at a fintech company. "
    "You are direct, opinionated, and intolerant of vague advice."
)
constraints = (
    "Respond with exactly four bullet points. "
    "Each bullet is one sentence. "
    "Audience: a junior data scientist about to run their first A/B test."
)
negatives = (
    "Do not start with an affirmation. "
    "Do not write an intro or conclusion paragraph. "
    "Do not give generic advice — every point must be actionable."
)
user_message = (
    "List the four most common mistakes junior data scientists make "
    "when designing an A/B test for a product feature."
)

system_prompt = f"{role}\n{constraints}\n{negatives}"

resp = router.generate_response(user_message, model=MODEL, system=system_prompt, max_tokens=400)

print("System prompt:")
print(system_prompt)
print()
print("User message:")
print(user_message)
print()
print("=" * 55)
print("Response:")
print(resp.text)
print()
print(f"Input tokens : {resp.input_tokens}")
print(f"Output tokens: {resp.output_tokens}")
print("All five dimensions active — no examples needed.")

## Visualising prompt quality — what changes across the five dimensions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Score three example prompts across the five zero-shot dimensions (0=absent, 1=present)
dimensions = ["Specificity", "Task\nFraming", "Role /\nPersona", "Constraints", "Negative\nInstructions"]

prompts = {
    "Bare prompt\n(tell me about X)":            [0.1, 0.2, 0.0, 0.0, 0.0],
    "Partially structured\n(explain X clearly)": [0.5, 0.7, 0.0, 0.3, 0.0],
    "Five-dimension\n(this notebook's template)": [1.0, 1.0, 1.0, 1.0, 1.0],
}

x = np.arange(len(dimensions))
width = 0.25
colours = ["tomato", "steelblue", "seagreen"]

fig, ax = plt.subplots(figsize=(10, 5))

for i, (label, scores) in enumerate(prompts.items()):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, scores, width, label=label, color=colours[i], alpha=0.88)

ax.set_ylabel("Dimension present (0 = absent, 1 = fully specified)")
ax.set_title("Zero-shot prompt quality across five dimensions")
ax.set_xticks(x)
ax.set_xticklabels(dimensions, fontsize=10)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=9)
ax.axhline(1.0, color="grey", linewidth=0.6, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("A bare prompt misses all five dimensions — the model fills gaps with its own defaults.")
print("Each dimension you fill in reduces ambiguity and moves the output closer to what you want.")
print("You rarely need all five — but knowing which one is missing is the skill.")

## Measuring the impact — output quality across prompt versions

Let's run the same underlying task with three progressively better prompts and measure the outputs by length and signal density (ratio of content words to total words — a rough proxy for filler).

In [ ]:
import re
import matplotlib.pyplot as plt

FILLER = {
    "certainly", "great", "absolutely", "of course", "sure", "indeed",
    "it's worth noting", "it is worth noting", "in conclusion", "to summarize",
    "in summary", "as you can see", "hope this helps", "feel free",
}

def filler_ratio(text: str) -> float:
    words = text.lower().split()
    filler_count = sum(1 for w in words if w.strip('.,!?') in FILLER)
    return filler_count / max(len(words), 1)

task = "Explain why database indexing matters for application performance."

prompt_versions = [
    {
        "label":  "Bare",
        "system": "You are a helpful assistant.",
        "user":   task,
    },
    {
        "label":  "Framed",
        "system": "You are a backend engineer. Be direct and avoid jargon where possible.",
        "user":   task + " Keep it under 80 words.",
    },
    {
        "label":  "Five-dimension",
        "system": (
            "You are a senior backend engineer explaining to a junior developer who just joined the team. "
            "Be direct and concrete — use a one-sentence analogy, then one practical consequence. "
            "Do not open with an affirmation. Do not add a concluding paragraph."
        ),
        "user": "In exactly three sentences, explain why database indexing matters for application performance.",
    },
]

results = []
for pv in prompt_versions:
    resp = router.generate_response(pv["user"], model=MODEL, system=pv["system"], max_tokens=300)
    text = resp.text
    results.append({
        "label":         pv["label"],
        "text":          text,
        "word_count":    len(text.split()),
        "filler_ratio":  filler_ratio(text),
        "input_tokens":  resp.input_tokens,
        "output_tokens": resp.output_tokens,
    })

print("Responses:")
for r in results:
    print(f"\n[{r['label']}]  ({r['word_count']} words, {r['output_tokens']} output tokens)")
    print(r['text'])

print()
print(f"{'Prompt':<18}  {'Words':>6}  {'Output tokens':>14}  {'Filler ratio':>13}")
print("-" * 56)
for r in results:
    print(f"{r['label']:<18}  {r['word_count']:>6}  {r['output_tokens']:>14}  {r['filler_ratio']:>12.3f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels       = [r["label"] for r in results]
word_counts  = [r["word_count"] for r in results]
filler_pcts  = [r["filler_ratio"] * 100 for r in results]

x = np.arange(len(labels))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.bar(labels, word_counts, color=["tomato", "steelblue", "seagreen"], alpha=0.88)
ax1.set_title("Output length (words)")
ax1.set_ylabel("Word count")
for i, v in enumerate(word_counts):
    ax1.text(i, v + 2, str(v), ha='center', fontsize=11, fontweight='bold')

ax2.bar(labels, filler_pcts, color=["tomato", "steelblue", "seagreen"], alpha=0.88)
ax2.set_title("Filler word ratio (lower is better)")
ax2.set_ylabel("% filler words")
for i, v in enumerate(filler_pcts):
    ax2.text(i, v + 0.05, f"{v:.1f}%", ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("A better-structured prompt produces shorter, denser responses.")
print("Fewer words with lower filler ratio = more signal per token.")
print("This matters even more when you process thousands of responses in a pipeline.")

## Key takeaways

- **Zero-shot means no examples** — it does not mean unstructured. You can get production-quality output with zero examples if your prompt is well-specified.
- **Specificity** prevents the model from defaulting to the average answer. Precise prompts cut the output space before the model starts generating.
- **Task framing** — the verb you use first — sets the cognitive mode for the entire response. Choose it deliberately.
- **Role assignment** loads a bundle of behaviours: vocabulary, depth, tone, and assumed audience. A specific role is worth more than a long list of adjectives.
- **Constraints** are information, not restrictions. Length, format, audience, and scope constraints replace ambiguity with a clear success target.
- **Negative instructions** block specific failure modes — filler openers, unnecessary summaries, vague hedging — that positive instructions alone do not prevent.
- **All five dimensions together** produce responses that are shorter, denser, and more directly useful — with no examples at all.

---

Next up: **Few-Shot Prompting** — when zero-shot is not enough, a small number of well-chosen examples can bridge the gap between what you can describe and what you actually need.